In [36]:
import tensorflow
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense,SimpleRNN

In [37]:
import numpy as np

In [50]:
import numpy as np
import keras
from keras.datasets import imdb
from keras.utils import pad_sequences

vocab_size = 10000
max_length = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print(len(X_train))   # 25000
print(len(X_test))    # 25000
print(X_train[0][:20])
print(y_train[0])

25000
25000
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
1


In [51]:
X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print(X_train.shape)
print(X_test.shape)


(25000, 200)
(25000, 200)


In [56]:
from keras import Sequential
from keras.layers import Input, Embedding, SimpleRNN, Dense

model = Sequential([
    Input(shape=(200,)),

    Embedding(
        input_dim=10000,
        output_dim=32,
        mask_zero=True
    ),

    SimpleRNN(32),

    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 322,113 (1.23 MB)

 Trainable params: 322,113 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [57]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.5900 - loss: 0.6605 - val_accuracy: 0.7786 - val_loss: 0.4929
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.7456 - loss: 0.5146 - val_accuracy: 0.7894 - val_loss: 0.4636
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.8794 - loss: 0.2899 - val_accuracy: 0.8036 - val_loss: 0.4719


In [61]:
word_index = imdb.get_word_index()

print(word_index["movie"])
print(word_index["amazing"])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
17
477


In [62]:
import re

def predict_sentiment(review):
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", review.lower())

    sequence = [1]  # START token

    for word in words:
        index = word_index.get(word)

        if index is None or index + 3 >= vocab_size:
            sequence.append(2)  # Unknown word
        else:
            sequence.append(index + 3)

    padded = pad_sequences(
        [sequence],
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    prediction = float(model.predict(padded, verbose=0)[0][0])

    print("Positive probability:", prediction)
    print("Sentiment:", "Positive" if prediction >= 0.5 else "Negative")

In [63]:
predict_sentiment("This movie was absolutely amazing and wonderful")



Positive probability: 0.8952572345733643
Sentiment: Positive


In [64]:
predict_sentiment("This movie was boring and terrible")

Positive probability: 0.030239509418606758
Sentiment: Negative


In [69]:

predict_sentiment("I really enjoyed tho it was a boring and dont watch this anyway and bad")

Positive probability: 0.21654552221298218
Sentiment: Negative
